# Smart Contract Automation — DonSheet to Adobe Sign

This notebook creates **one populated PDF per contracted course** from a DonSheet, then creates one Adobe Acrobat Sign agreement per PDF. Adobe Sign routes the agreement in this fixed order: instructor, chair, dean, VP Finance, then HR.

The PDF-generation code is the same implementation used by the Streamlit **Shari → Create contract PDFs – based on DonSheet** workflow. The sending cells default to Adobe Sign drafts; they do not email anyone until `SEND_MODE` is changed to `"send"` and both confirmation prompts are answered `YES`.

In [ ]:
from pathlib import Path
import os
import sys
import re
import time
import json
import pandas as pd
import requests

# Locate the repository whether Jupyter was started in the repo root or in jupyter_notebooks/.
_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_here, *_here.parents) if (p / 'programs' / 'app').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Start this notebook inside the SDATS_SmartSeminary repository.')

sys.path.insert(0, str(PROJECT_ROOT / 'programs'))
from app.contracts_from_donsheet import (
    extract_contract_rows_from_donsheet, resolve_app_paths, generate_contract_pdfs, sanitize_filename,
)

pd.set_option('display.max_columns', None)
print('Project root:', PROJECT_ROOT)

## 1. Select the DonSheet and output folder

Place the completed DonSheet in `Step03_input_contract-production_DonSheets_filled/` (or set `DONSHEET_PATH` explicitly). Only rows whose **load/contract** value is `contract` are processed.

In [ ]:
INPUT_DIR = PROJECT_ROOT / 'Step03_input_contract-production_DonSheets_filled'
OUTPUT_DIR = PROJECT_ROOT / 'Step03_output_contract-production_PDF-CONTRACTS_filled'
INPUT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Set a concrete path here when more than one DonSheet is in INPUT_DIR.
DONSHEET_PATH = None

if DONSHEET_PATH is None:
    candidates = sorted(p for p in INPUT_DIR.glob('*.xlsx') if not p.name.startswith(('._', '~$')))
    if len(candidates) != 1:
        raise FileNotFoundError(
            f'Expected exactly one .xlsx file in {INPUT_DIR}; found {len(candidates)}. '
            'Put one file there or set DONSHEET_PATH explicitly.'
        )
    DONSHEET_PATH = candidates[0]
else:
    DONSHEET_PATH = Path(DONSHEET_PATH)

PDF_PATHS = resolve_app_paths(PROJECT_ROOT)
print('DonSheet:', DONSHEET_PATH)
print('Standard PDF template:', PDF_PATHS.standard_template.name)
print('Intensive PDF template:', PDF_PATHS.intensive_template.name)
print('PDF output:', OUTPUT_DIR)

## 2. Extract and validate contracted-course rows

The DonSheet fields used for routing are `Instructor Email {Instr Email}`, `chair email`, `dean email`, `VP finance email`, and `HR email`. The notebook stops before creating any Adobe Sign agreement if a required routing address is missing.

In [ ]:
df_courses = extract_contract_rows_from_donsheet(DONSHEET_PATH)
if df_courses.empty:
    raise ValueError('No DonSheet rows marked as contract were found.')

REQUIRED_SIGNER_COLUMNS = ['Email', 'ChairEmail', 'DeanEmail', 'VPFinanceEmail', 'HRemail']
def clean_email(value):
    text = '' if value is None else str(value).strip()
    return '' if text.lower() == 'nan' else text

missing_columns = [column for column in REQUIRED_SIGNER_COLUMNS if column not in df_courses.columns]
if missing_columns:
    raise KeyError(f'DonSheet extraction did not produce signer columns: {missing_columns}')

for column in REQUIRED_SIGNER_COLUMNS:
    df_courses[column] = df_courses[column].map(clean_email)

missing = df_courses[df_courses[REQUIRED_SIGNER_COLUMNS].eq('').any(axis=1)]
if not missing.empty:
    display(missing[['Instructor', 'Semester', 'Year', 'CourseRaw', *REQUIRED_SIGNER_COLUMNS]])
    raise ValueError('Fill every required signer-email field in the DonSheet, then rerun this cell.')

display(df_courses[['Instructor', 'Semester', 'Year', 'CourseRaw', 'CourseSection', 'Email', 'ChairEmail', 'DeanEmail', 'VPFinanceEmail', 'HRemail']])
print(f'Contract PDFs to create: {len(df_courses)}')

## 3. Create PDF contracts

The PDF form fields are populated and flattened before upload. `Source Row` is included in each filename to keep course contracts distinct. Clear the output directory yourself before rerunning if you do not want old PDFs included in the Adobe Sign batch.

In [ ]:
generated_contracts = generate_contract_pdfs(
    df_courses,
    PDF_PATHS,
    OUTPUT_DIR,
    filename_prefix=f'{sanitize_filename(DONSHEET_PATH.stem)}__',
    source_file=DONSHEET_PATH.name,
)

df_generated = pd.DataFrame([
    {
        'pdf_file': contract.filename,
        'pdf_path': str(contract.path),
        'contract_type': contract.contract_type,
        'instructor': contract.instructor,
        'course_code': contract.course_code,
    }
    for contract in generated_contracts
])
display(df_generated)
print(f'Created {len(generated_contracts)} PDFs in {OUTPUT_DIR}')

## 4. Adobe Acrobat Sign setup

Create an Adobe Sign API application and provide an access token with agreement send/read permissions. Set `ADOBE_SIGN_ACCESS_TOKEN` in the notebook environment instead of hard-coding it. The `/baseUris` call determines the correct Adobe Sign shard for that token.

For signatures in fixed locations, prepare the PDF templates in Acrobat Sign with Adobe Sign text tags or a form-field layer that is attached to the source PDF before this workflow. The filled PDFs already contain the contract values; Adobe Sign only needs the signature/initial fields. If the account's agreement settings add a signature block automatically, no additional form-field layer is required.

In [ ]:
ADOBE_SIGN_ACCESS_TOKEN = os.getenv('ADOBE_SIGN_ACCESS_TOKEN', '')
if not ADOBE_SIGN_ACCESS_TOKEN:
    raise RuntimeError(
        'Set ADOBE_SIGN_ACCESS_TOKEN in the Jupyter environment, then rerun this cell. '
        'Do not paste a production token into the notebook.'
    )

class AdobeSignClient:
    """Small REST v6 client for upload, draft, and send operations."""
    GLOBAL_BASE_URI = 'https://api.na1.adobesign.com/api/rest/v6'

    def __init__(self, access_token):
        self.session = requests.Session()
        self.session.headers.update({'Authorization': f'Bearer {access_token}'})
        response = self.session.get(f'{self.GLOBAL_BASE_URI}/baseUris', timeout=30)
        self._raise_for_error(response)
        access_point = response.json()['apiAccessPoint'].rstrip('/')
        self.base_uri = access_point if access_point.endswith('/api/rest/v6') else access_point + '/api/rest/v6'

    @staticmethod
    def _raise_for_error(response):
        try:
            response.raise_for_status()
        except requests.HTTPError as exc:
            raise RuntimeError(f'Adobe Sign API error: {response.status_code} {response.text}') from exc

    def upload_pdf(self, pdf_path):
        with Path(pdf_path).open('rb') as pdf_file:
            response = self.session.post(
                f'{self.base_uri}/transientDocuments',
                files={'File': (Path(pdf_path).name, pdf_file, 'application/pdf')},
                headers={'File-Name': Path(pdf_path).name},
                timeout=120,
            )
        self._raise_for_error(response)
        return response.json()['transientDocumentId']

    def create_agreement(self, transient_document_id, agreement_name, participant_sets, state):
        payload = {
            'fileInfos': [{'transientDocumentId': transient_document_id}],
            'name': agreement_name,
            'participantSetsInfo': participant_sets,
            'signatureType': 'ESIGN',
            'state': state,
        }
        response = self.session.post(f'{self.base_uri}/agreements', json=payload, timeout=60)
        self._raise_for_error(response)
        return response.json()['id']

    def agreement_status(self, agreement_id):
        response = self.session.get(f'{self.base_uri}/agreements/{agreement_id}', timeout=30)
        self._raise_for_error(response)
        return response.json().get('status', '')

    def send_draft(self, agreement_id):
        response = self.session.put(
            f'{self.base_uri}/agreements/{agreement_id}/state',
            json={'state': 'IN_PROCESS'},
            timeout=60,
        )
        self._raise_for_error(response)

adobe = AdobeSignClient(ADOBE_SIGN_ACCESS_TOKEN)
print('Authenticated to Adobe Sign:', adobe.base_uri)

## 5. Build the routing manifest

Adobe Sign uses `order` to route each participant set sequentially. Each course produces its own agreement, so a problem with one row is recorded in the manifest without preventing the remaining contracts from being created.

In [ ]:
def participant_sets_from_row(row):
    # The order values are the workflow: Adobe Sign emails the next set only after the previous one completes.
    return [
        {'memberInfos': [{'email': row['Email']}], 'order': 1, 'role': 'SIGNER', 'name': 'Instructor'},
        {'memberInfos': [{'email': row['ChairEmail']}], 'order': 2, 'role': 'SIGNER', 'name': 'Chair'},
        {'memberInfos': [{'email': row['DeanEmail']}], 'order': 3, 'role': 'SIGNER', 'name': 'Dean'},
        {'memberInfos': [{'email': row['VPFinanceEmail']}], 'order': 4, 'role': 'SIGNER', 'name': 'VP Finance'},
        {'memberInfos': [{'email': row['HRemail']}], 'order': 5, 'role': 'SIGNER', 'name': 'Human Resources'},
    ]

# generate_contract_pdfs retains the course ordering, so pair each PDF with its originating DonSheet row.
if len(generated_contracts) != len(df_courses):
    raise RuntimeError('PDF count differs from extracted DonSheet row count; do not send this batch.')

send_manifest = df_courses.reset_index(drop=True).copy()
send_manifest['pdf_path'] = [str(contract.path) for contract in generated_contracts]
send_manifest['contract_type'] = [contract.contract_type for contract in generated_contracts]
send_manifest['agreement_name'] = [f'SEM Contract — {contract.instructor} — {contract.course_code}' for contract in generated_contracts]
display(send_manifest[['agreement_name', 'pdf_path', 'Email', 'ChairEmail', 'DeanEmail', 'VPFinanceEmail', 'HRemail']])

## 6. Create Adobe Sign drafts or send agreements

Leave `SEND_MODE = 'draft'` for the first run. Adobe Sign drafts are visible to the sender but do not email signers. After checking the PDFs, recipient order, and signature placement in Adobe Sign, use `SEND_MODE = 'send'`. This sends new agreements immediately; it does **not** send existing drafts.

In [ ]:
SEND_MODE = 'draft'  # 'draft' is safe; change to 'send' only after review.
if SEND_MODE not in {'draft', 'send'}:
    raise ValueError("SEND_MODE must be 'draft' or 'send'.")

if SEND_MODE == 'send':
    print('IMPORTANT: Every new agreement in this run will be emailed for signature.')
    if input('Type YES to confirm the PDFs and signer order were reviewed: ').strip() != 'YES':
        raise SystemExit('No Adobe Sign agreements were sent.')
    if input('Type YES again to send this batch: ').strip() != 'YES':
        raise SystemExit('No Adobe Sign agreements were sent.')

results = []
for _, row in send_manifest.iterrows():
    result = {
        'pdf_file': Path(row['pdf_path']).name,
        'agreement_name': row['agreement_name'],
        'mode': SEND_MODE,
        'agreement_id': '',
        'initial_status': '',
        'error': '',
    }
    try:
        transient_document_id = adobe.upload_pdf(row['pdf_path'])
        state = 'DRAFT' if SEND_MODE == 'draft' else 'IN_PROCESS'
        agreement_id = adobe.create_agreement(
            transient_document_id=transient_document_id,
            agreement_name=row['agreement_name'],
            participant_sets=participant_sets_from_row(row),
            state=state,
        )
        result['agreement_id'] = agreement_id
        # Agreement creation is asynchronous; this is informative, not a wait-for-signature loop.
        time.sleep(1)
        result['initial_status'] = adobe.agreement_status(agreement_id)
    except Exception as exc:
        result['error'] = f'{type(exc).__name__}: {exc}'
    results.append(result)

df_adobe_log = pd.DataFrame(results)
display(df_adobe_log)
log_path = OUTPUT_DIR / 'adobe_sign_agreement_log.csv'
df_adobe_log.to_csv(log_path, index=False)
print('Saved Adobe Sign log:', log_path)

## 7. Review failures

The CSV log preserves the Adobe Sign agreement ID for each successfully created agreement. Use the ID to inspect an individual agreement in Adobe Sign or to add future status/reminder automation.

In [ ]:
failures = df_adobe_log[df_adobe_log['error'].ne('')]
if failures.empty:
    print('No API failures were recorded.')
else:
    display(failures)